# 06 - Comparison report and uncertainty analysis

**Settings:** Accelerator = **None (CPU)**, Internet = On. Run with **Save Version > Save & Run All (Commit)**,
not the editor's Run All: a committed run keeps going when the laptop is closed and its output is saved.

**Inputs** (Add Input > Your Work > Notebooks):
1. `02_prepare_test` (ufm-test)
2. `03_train_core`
3. `04_train_ablation`
4. `05_baselines`

**Output:** `report/` (tables, bar chart, uncertainty, maps)  |  **Approx. time:** 15-30 min

Collects every run from the attached notebooks, checks nothing is missing, then writes the comparison table (seed mean +- sd, Jubba_2 excluded), the uncertainty analysis and the maps.

In [ ]:
# ---- Settings -------------------------------------------------------------
GITHUB_REPO = "https://github.com/mhdnasim6-boop/urban-flood-mapping.git"
BRANCH = "main"
# ---------------------------------------------------------------------------
# Private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets).
# Optional: a secret named HF_TOKEN (Hugging Face read token) avoids download rate limits.
import glob, os, subprocess
from pathlib import Path

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

url = GITHUB_REPO
if secret("GITHUB_TOKEN"):
    url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
subprocess.run(["rm", "-rf", "/tmp/ufm"])
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/tmp/ufm"], check=True)
os.chdir("/tmp/ufm")
print("code version:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_input(name, notebook):
    """Locate an attached input folder or stop immediately with a clear message."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise SystemExit(f"STOP: input '{name}' not found. Add the output of notebook {notebook} "
                         "(Add Input > Your Work > Notebooks) and run again.")
    print(f"{name}: {hits[0]}")
    return hits[0]

In [ ]:
TEST = find_input("ufm-test", "02_prepare_test")
EXCLUDE = ["20231201_Jubba_2"]   # too few reference flood pixels inside the published chips
EXPECTED = ["D_full", "B5_authors_baseline", "A_raw", "D_full_s1", "A_raw_s1", "D_full_s2", "A_raw_s2",
            "B_raw_phys", "C_raw_aux", "E_no_coherence", "F_no_vh",
            "baselines/B1", "baselines/B2", "baselines/B3", "baselines/B4"]

# link every run folder from the attached notebooks into one place (nothing is copied)
RUNS = Path("/tmp/runs"); (RUNS / "baselines").mkdir(parents=True, exist_ok=True)
for src in glob.glob("/kaggle/input/**/runs", recursive=True):
    for d in Path(src).iterdir():
        if d.name == "baselines":
            for b in d.iterdir():
                if not (RUNS / "baselines" / b.name).exists():
                    (RUNS / "baselines" / b.name).symlink_to(b)
        elif d.is_dir() and not (RUNS / d.name).exists():
            (RUNS / d.name).symlink_to(d)
missing = [e for e in EXPECTED if not (RUNS / e / "test" / "test_metrics.csv").exists()]
if not (RUNS / "D_full" / "test_mc" / "test_metrics.csv").exists():
    missing.append("D_full/test_mc (MC-dropout maps)")
if missing:
    raise SystemExit(f"STOP: missing runs {missing}. Attach notebooks 03, 04 and 05 (after they finish).")
print("all runs present:", sorted(p.name for p in RUNS.iterdir()))

In [ ]:
!pip install -q rasterio tabulate 2>/dev/null

In [ ]:
!python scripts/uncertainty_analysis.py --maps {RUNS}/D_full/test_mc --test-root {TEST} --out /kaggle/working/report --exclude-events {' '.join(EXCLUDE)}

In [ ]:
!python scripts/summarize.py --runs {RUNS} --test-root {TEST} --out /kaggle/working/report --exclude-events {' '.join(EXCLUDE)}

In [ ]:
!cp /tmp/runs/baselines/settings.json /kaggle/working/report/baseline_settings.json 2>/dev/null; true
!for r in /tmp/runs/*/history.csv; do cp $r /kaggle/working/report/history_$(basename $(dirname $r)).csv; done
from IPython.display import Markdown, Image, display
for md in ["comparison.md", "uncertainty.md"]:
    display(Markdown(open(f"/kaggle/working/report/{md}").read()))
for p in ["f1_bars.png", "uncertainty.png"] + sorted(p.name for p in Path("/kaggle/working/report").glob("maps_*.png")):
    display(Image(f"/kaggle/working/report/{p}"))